# Statistical Analysis: Maji Ndogo Agricultural Dataset

## Objective

This notebook performs statistical analysis on the cleaned Maji Ndogo dataset.

The goal is to understand:
- Distribution of variables
- Central tendency and spread
- Feature relationships
- Statistical significance
- Regression assumptions

The target variable is:

**Standard_yield**

# Import libraries

In [ ]:
cd ..

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.pipeline import create_final_dataset
from src.exploratory_data_analysis import ExploratoryDataAnalysis

In [ ]:
EDA = ExploratoryDataAnalysis()
df_original = EDA.get_data()
df = df_original.copy()



# Data and Descriptive statistics

In [ ]:
EDA.quality_assessment()

In [ ]:
df.info()

In [ ]:
EDA.summary_statistics()

# Target Variable Analysis

In [ ]:
EDA.plot_numerical_distribution("Standard_yield")


In [ ]:
EDA.single_num_variable_desc_statistics("Standard_yield")

The distribution of Standard_yield is approximately normal with mild right skew, centered at mean=0.534 and median=0.529. The standard deviation of 0.112 indicates moderate variability most farms yield between 0.42 and 0.65, but the full range spans 0.17 to 0.90. There are no extreme outliers. 

# Measure of Skewness and outliers

In [ ]:
numeric_cols =EDA.get_numeric_columns()

In [ ]:
def numeric_feature_distribution(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates the skewness for all numeric columns.
    """
    results = []

    for col in numeric_cols:
        skew_value = df[col].skew()

        try:
            skew_value = float(skew_value)
        except (TypeError, ValueError):
            continue

        if skew_value <= -1:
            skew_interp = "Highly Left-Skewed"
        elif skew_value < -0.5:
            skew_interp = "Moderately Left-Skewed"
        elif skew_value <= 0.5:
            skew_interp = "Approximately Symmetric"
        elif skew_value <= 1:
            skew_interp = "Moderately Right-Skewed"
        else:
            skew_interp = "Highly Right-Skewed"

        results.append({
            "feature": col,
            "Skewness": round(skew_value, 3),
            "Skewness_Interpretation": skew_interp
        })

    return pd.DataFrame(results).sort_values(by="Skewness", ascending=False).reset_index(drop=True)

numeric_feature_distribution(df)

### Skewness Analysis Insight

Most numerical features exhibit approximately symmetric distributions, with skewness values between -0.5 and 0.5. The target variable, `Standard_yield`, is also fairly balanced with only slight positive skewness, which is suitable for regression modeling.

However, `Annual_yield`, `Slope`, `Plot_size`, and `Field_Pollution_Level` show high positive skewness, indicating the presence of long right tails and potential extreme values. These features will be further investigated using visualizations such as box plots to determine whether the skewness is caused by natural variations or potential outliers.


# Correlation Analysis

In [83]:
def target_correlation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns correlations of numerical features with the target variable.
    """
    results = []

    for col in numeric_cols:
        correlation = df[numeric_cols].corr()
        target_corr = (
            correlation["Standard_yield"]
            .sort_values(ascending=False)
            .to_frame()
        )

    return target_corr


target_correlation(df)

,Standard_yield
Standard_yield,1.000000
Annual_yield,0.220812
Station_Pollution_Level,0.164481
Min_temperature_C,0.144233
Elevation,0.129248
Longitude,0.085343
Soil_fertility,0.070205
Temperature,0.062516
Latitude,0.061724
Slope,0.056991


In [82]:
def correlation_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """
    This method returns the correlations of features predictors to target feature
    """
    correlation = pd.DataFrame(df[numeric_cols].corr())
    correlation["Standard_yield"].sort_values(ascending = False)
    return correlation
correlation_matrix(df)


,Elevation,Latitude,Longitude,Slope,Field_Rainfall,Min_temperature_C,Max_temperature_C,Ave_temps,Soil_fertility,pH,Field_Pollution_Level,Plot_size,Annual_yield,Standard_yield,Station_Pollution_Level,Station_Rainfall,Temperature
Elevation,1.000000,0.546737,0.345139,0.081837,-0.238518,0.956418,-0.605884,0.203061,-0.146844,-0.430787,0.285086,-0.032261,0.005033,0.129248,0.519142,-0.524937,-0.007350
Latitude,0.546737,1.000000,0.257728,0.103381,-0.756442,0.337544,-0.340948,-0.077763,-0.566838,-0.220201,0.289556,-0.064183,-0.045701,0.061724,0.475569,-0.799233,-0.184139
Longitude,0.345139,0.257728,1.000000,-0.078865,0.146665,0.398923,-0.223253,0.119329,0.071705,-0.315035,0.476562,0.065011,0.088318,0.085343,0.743867,0.113991,0.730206
Slope,0.081837,0.103381,-0.078865,1.000000,-0.131472,0.044370,-0.046567,-0.012279,0.550326,-0.087969,0.010879,-0.603773,-0.571007,0.056991,0.073452,-0.128668,-0.120772
Field_Rainfall,-0.238518,-0.756442,0.146665,-0.131472,1.000000,0.052469,0.155806,0.233366,0.752914,0.025182,-0.198268,0.092198,0.098114,0.039217,-0.184830,0.781365,0.490543
Min_temperature_C,0.956418,0.337544,0.398923,0.044370,0.052469,1.000000,-0.576769,0.278994,0.072797,-0.435366,0.235310,-0.005503,0.034617,0.144233,0.479000,-0.306884,0.139173
Max_temperature_C,-0.605884,-0.340948,-0.223253,-0.046567,0.155806,-0.576769,1.000000,0.623555,0.099502,0.255306,-0.161467,0.018017,-0.011637,-0.111649,-0.324746,0.323234,0.002282
Ave_temps,0.203061,-0.077763,0.119329,-0.012279,0.233366,0.278994,0.623555,1.000000,0.186633,-0.116527,0.035383,0.015914,0.019449,0.006786,0.076657,0.086281,0.135871
Soil_fertility,-0.146844,-0.566838,0.071705,0.550326,0.752914,0.072797,0.099502,0.186633,1.000000,-0.035415,-0.160151,-0.320810,-0.294160,0.070205,-0.106391,0.571569,0.332629
pH,-0.430787,-0.220201,-0.315035,-0.087969,0.025182,-0.435366,0.255306,-0.116527,-0.035415,1.000000,-0.140758,0.055303,0.003464,-0.196613,-0.341531,0.151004,-0.153293


In [ ]:
def correlation_heatmap(df: pd.DataFrame) -> None:
    corr_matrix = df[numeric_cols].corr()
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
    plt.title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()

correlation_heatmap(df)

### Correlation Analysis Insight

The correlation analysis shows that most features have weak linear relationships with the target variable, `Standard_yield`. The strongest positive correlation with yield is observed in `Annual_yield` (0.221), while `Field_Pollution_Level` shows the strongest negative relationship (-0.286). This suggests that `Standard_yield` is likely influenced by multiple factors rather than a single dominant feature.

The analysis also reveals strong correlations between some predictor variables, such as `Plot_size` and `Annual_yield` (0.949), and between temperature-related variables such as `Elevation` and `Min_temperature_C` (0.956). These strong relationships may indicate potential multicollinearity, which will be investigated further using Variance Inflation Factor (VIF) before model training.


# Statistical Significance (Pearson Test)

In [78]:
def pearson_test(df: pd.DataFrame):
    """
    This method determines if a a relationship is a statistical significant
    """
    corr, p_value = stats.pearsonr(df["Plot_size"], df["Standard_yield"])
    return corr, p_value

corr, p_value = pearson_test(df)

print("Correlation:", corr)
print("p-value:", p_value)

Correlation: -0.017014248418182613
p-value: 0.20083947341956868


In [ ]:
def pearson_test(df: pd.DataFrame, feature_cols: list = None, target_col: str = "Standard_yield") -> pd.DataFrame:
    """
    Performs Pearson correlation test between multiple features and target variable.
    """
    if feature_cols is None:
        feature_cols = [col for col in numeric_cols if col != target_col]
    
    results = []
    
    for feature in feature_cols:
        clean_df = df[[feature, target_col]].dropna()

        if len(clean_df) < 3:
            results.append({
                'Feature': feature,
                'Correlation': None,
                'P-Value': None,
                'Significant': None,
                'Interpretation': 'Insufficient data'
            })
            continue

        # pearson test
        corr, p_value = stats.pearsonr(clean_df[feature], clean_df[target_col])
        

        significant = p_value < 0.05
        
        # Interpretation
        if significant:
            if abs(corr) > 0.7:
                direction = 'Strong positive' if corr > 0 else 'Strong negative'
            elif abs(corr) > 0.3:
                direction = 'Moderate positive' if corr > 0 else 'Moderate negative'
            else:
                direction = 'Weak positive' if corr > 0 else 'Weak negative'
            interpretation = f"{direction} correlation"
        else:
            interpretation = 'Not significant'
        
        results.append({
            'Feature': feature,
            'Correlation': round(corr, 4),
            'P-Value': round(p_value, 4),
            'Significant': 'Yes' if significant else ' No',
            'Interpretation': interpretation
        })
    
    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values('Correlation', ascending=False, key=abs, na_position='last')
    
    return result_df

results = pearson_test(df, target_col="Standard_yield")
print(results.to_string(index=False))

                Feature  Correlation  P-Value Significant            Interpretation
  Field_Pollution_Level      -0.2858   0.0000         Yes Weak negative correlation
           Annual_yield       0.2208   0.0000         Yes Weak positive correlation
                     pH      -0.1966   0.0000         Yes Weak negative correlation
Station_Pollution_Level       0.1645   0.0000         Yes Weak positive correlation
      Min_temperature_C       0.1442   0.0000         Yes Weak positive correlation
              Elevation       0.1292   0.0000         Yes Weak positive correlation
      Max_temperature_C      -0.1116   0.0000         Yes Weak negative correlation
              Longitude       0.0853   0.0000         Yes Weak positive correlation
         Soil_fertility       0.0702   0.0000         Yes Weak positive correlation
            Temperature       0.0625   0.0000         Yes Weak positive correlation
               Latitude       0.0617   0.0000         Yes Weak positive corr